## Apresentação

Notebook destinado à implementação de um chatbot com memória e contextualizador de mensagens, para que consiga garantir uma experiência fluida ao usuário e otimização do algoritmo de busca do RAG. 

### Library

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import getpass
import logging
import os
from typing import Dict, List

from features.clean_memory import CleanMemory
from IPython.display import Markdown
from langchain.chains import ConversationChain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain
from langchain.memory import (ConversationBufferMemory,
                              ConversationSummaryBufferMemory,
                              ConversationSummaryMemory)
from langchain.schema import Document
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.chat_history import (BaseChatMessageHistory,
                                         InMemoryChatMessageHistory)
from langchain_core.embeddings import Embeddings
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import HumanMessage, trim_messages
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (ChatPromptTemplate, MessagesPlaceholder,
                                    PromptTemplate)
from langchain_core.retrievers import BaseRetriever
from langchain_core.runnables import Runnable, RunnableBranch, RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.vectorstores import InMemoryVectorStore, VectorStore
from langchain_groq import ChatGroq
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from prompt.chat_system_message import system_message
from prompt.check_context import check_context_prompt
from prompt.contextualize_message import contextualize_prompt
from prompt.simple_system_message import simple_system_prompt
from prompt.system_message import system_prompt

### Inicializando a LLM

In [ ]:
# API reference : ...

os.environ["GROQ_API_KEY"]=getpass.getpass("Your API Key: ")

In [17]:
llama = "llama3-70b-8192"
deepseek = "deepseek-r1-distill-llama-70b"

llm = ChatGroq(
    model = llama, 
    temperature = 0
)

llm.invoke("Olá, tudo bem ?").content

2025-06-08 22:34:07 - DEBUG - Request options: {'method': 'post', 'url': '/openai/v1/chat/completions', 'files': None, 'json_data': {'messages': [{'role': 'user', 'content': 'Olá, tudo bem ?'}], 'model': 'llama3-70b-8192', 'n': 1, 'stop': None, 'stream': False, 'temperature': 1e-08}}
2025-06-08 22:34:07 - DEBUG - Sending HTTP Request: POST https://api.groq.com/openai/v1/chat/completions
2025-06-08 22:34:07 - DEBUG - connect_tcp.started host='api.groq.com' port=443 local_address=None timeout=None socket_options=None
2025-06-08 22:34:07 - DEBUG - connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000013F816BCF50>
2025-06-08 22:34:07 - DEBUG - start_tls.started ssl_context=<ssl.SSLContext object at 0x0000013F81767DA0> server_hostname='api.groq.com' timeout=None
2025-06-08 22:34:07 - DEBUG - start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000013F816CC690>
2025-06-08 22:34:07 - DEBUG - send_request_headers.started request=<Req

'Olá! Sim, tudo bem, obrigado! E você?'

### Embedding

In [5]:
%%time

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

CPU times: total: 7.22 s
Wall time: 35.5 s


### Formando a base de conhecimento

Base de conhecimento, também conhecida como knowledge base se refere a uma fonte de informação a partir da qual o modelo utiliza para responder o usuário, visando garantir um incremento da qualidade de resposta, proporcionando uma não dependência do pré-treinamento dos modelos de LLM. 

In [6]:
%%time

"""
Elaborando os métodos utilizados para o modelo possuir
a sua base de conhecimento.  
"""

loader = PyPDFLoader("../data/Review of AI and Mental Health.pdf").load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size         = 500, 
    chunk_overlap      = 50, 
    length_function    = len,
    separators         = ["", " ", ".", "\n", "\n\n"],
    is_separator_regex = False
).split_documents(loader)    

retriever = InMemoryVectorStore.from_documents( 
    documents = text_splitter,
    embedding = embeddings
).as_retriever(search_kwargs={"k": 3})

CPU times: total: 1min 59s
Wall time: 35.8 s


### Chatbot 

In [7]:
# Configura o logging para exibir mensagens no console
logging.basicConfig(
    level=logging.DEBUG,  # Mostra logs DEBUG, INFO, WARNING, ERROR e CRITICAL
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

In [8]:
class Bimo:
    """
    Conversational RAG (Retrieval-Augmented Generation) system for handling 
    conversational queries with context-aware retrieval.
    """
    def __init__(
            self, 
            llm: BaseChatModel, 
            system_message: str,
            check_context_prompt: PromptTemplate,
            contextualizer_prompt: PromptTemplate, 
            retriever: VectorStore, 
            memory: ChatMessageHistory,
            include_memory: bool = True, 
            max_messages: int = 5
        ) -> None:
        """
        Initializes the ConversationalRag instance.

        Args:
            llm (BaseChatModel): The language model used for response generation.
            system_message (str): The system-level instruction message.
            contextualize_message (str): The message to provide context-aware queries.
            embedding (Embeddings): The embedding model used for document retrieval.
            memory (ChatMessageHistory): The chat history manager.
            documents (List[Document]): A list of documents to be processed and retrieved.
        """
        self.llm                   = llm 
        self.system_message        = system_message
        self.check_context_prompt  = check_context_prompt
        self.contextualizer_prompt = contextualizer_prompt
        self.retriever             = retriever
        self.memory                = memory
        self.include_memory        = include_memory
        self.max_messages          = max_messages
        self.store                 = {}
        self.logger                = logging.getLogger(__name__)
        self.clean_memory          = CleanMemory(
            max_messages = self.max_messages,
            strategy     = "last",
            start_on     = "human"
        )
        self.__format_prompt()


    def get_session_history(self, session_id: str) -> BaseChatMessageHistory: 
        """ 
        Retrieves or initializes the chat history for a given session.

        Args:
            session_id (str): The unique identifier for the chat session.
        
        Returns:
            BaseChatMessageHistory: The chat history associated with the session.
        """ 
        if session_id not in self.store: 
            self.store[session_id] = self.memory
        self.logger.info(f"Memória: {self.store[session_id]}")
        return self.store[session_id]

    def __format_prompt(self) -> None: 
        """ 
        Formats the system and contextualization prompts for structured conversation handling.
        """ 

        self.__system_prompt = ChatPromptTemplate(
            [
                ("system", self.system_message),
                MessagesPlaceholder("chat_history"), 
                ("human", "{question}")
            ]
        )
    
    def check_context(self, query: str, chat_history: List[str]) -> str:
        """
        Determine whether the current query requires additional context from chat history.

        Args:
            query (str): The user question to evaluate.
            chat_history (List[str]): List of prior chat messages.

        Returns:
            str: Model response indicating whether context is needed (e.g., "Yes" or "No").
        """
        partial = self.check_context_prompt.partial(chat_history=chat_history)
        chain = partial | self.llm
        return chain.invoke({"question": query})

    def contextualize_question(self, query: str, chat_history: List[str]) -> List[str]:
        """
        Rephrase the query to include relevant context from the chat history.

        Args:
            query (str): The original user question.
            chat_history (List[str]): List of prior chat messages.

        Returns:
            List[str]: Contextualized query messages for downstream processing.
        """
        partial = self.contextualizer_prompt.partial(chat_history=chat_history)
        chain = partial | self.llm
        return chain.invoke({"question": query})

    def process_and_reformulate_memory(self, input: Dict[str, str]) -> None:
        """
        Check if the input should use memory context and reformulate it if needed.

        Args:
            input (Dict[str, str]): Dictionary with key "input" containing the user query.
        """
        response = self.check_context(
            query=input["question"],
            chat_history=self.memory.messages[-1]
        )
        self.logger.info(f"Context check result: {response}")

        if response in ("Yes", "Sim"):
            self.logger.info(f"{response}")
            reformulated = self.contextualize_question(
                query=input["question"],
                chat_history=self.memory.messages[-1]
            )
    
            self.logger.info(f"Contextualized message: {reformulated}")

    def update_memory(self, input: Dict[str, str]) -> None:
        """
        Add the latest human message to memory and enforce memory size limits.

        Args:
            input (Dict[str, str]): Dictionary with key "input" containing the user query.
        """
        self.memory.add_messages([HumanMessage(content=input["question"])])
        self.clean_memory.trim_messages(self.memory)

    def retrieve_document_as_a_list(self, input) -> List:
        """
        Retrieve relevant documents for the given input query.

        Accepts either:
            - a plain string (the query itself), or
            - a dict with "question" or "input" keys.

        Returns:
            List: Retrieved documents as a list.
        """
        if isinstance(input, str):
            query_text = input
        elif isinstance(input, dict):
            # tenta as duas chaves que você usa no pipeline
            query_text = input.get("question") or input.get("input")
            if query_text is None:
                raise ValueError("Esperava dicionário com chave 'question' ou 'input'")
        else:
            raise TypeError(f"Tipo de input inesperado: {type(input)}")

        return self.retriever.invoke(query_text)


    def retrieved_documents(self) -> Runnable:
        """
        Build a runnable pipeline to fetch documents, optionally using chat history branching.
             
        Returns:
            Runnable: A configured retrieval pipeline.
        """
        return RunnableBranch(
            (
                lambda x: not x.get("chat_history", False),
                RunnableLambda(lambda inputs: self.retrieve_document_as_a_list(inputs)),
            ),
            self.contextualizer_prompt | self.llm | StrOutputParser() |
            RunnableLambda(lambda inputs: self.retrieve_document_as_a_list(inputs)),
        ).with_config(run_name="chat_retriever_chain")

    def buid_conversational_chain(self) -> Runnable:
        """ 
        Builds the conversational RAG chain by combining retrieval and response generation.

        Returns:
            Runnable: A runnable chain for processing conversational queries.
        """  

        question_answer_chain = create_stuff_documents_chain(
            self.llm, 
            self.__system_prompt
        )

        rag_chain = create_retrieval_chain(
            self.retrieved_documents(), 
            question_answer_chain
        )

        return rag_chain

    def run(self, query: str) -> str:
        """ 
        Executes the RAG pipeline for a given query and returns the generated response.

        Args:
            query (str): The user input query.
        
        Returns:
            str: The generated response from the conversational model.
        """ 
        if self.include_memory and len(self.memory.messages) > 1:
            self.process_and_reformulate_memory({"question": query})

        conversational_rag_chain = RunnableWithMessageHistory(
            self.buid_conversational_chain(), 
            self.get_session_history, 
            input_messages_key   = "question", 
            history_messages_key = "chat_history", 
            output_messages_key  = "answer"
        )

        response = conversational_rag_chain.invoke(
            {"question": query}, 
            config={
                "configurable": {"session_id": 935}
            }
        )["answer"]
    
        self.logger.info(f"Input message: {query}")
        self.logger.info(f"Response: {response}")

        return response

In [9]:
bimo = Bimo(
    llm                   = llm, 
    system_message        = system_message, 
    check_context_prompt  = check_context_prompt, 
    contextualizer_prompt = contextualize_prompt, 
    memory                = InMemoryChatMessageHistory(), 
    retriever             = retriever, 
    include_memory        = True, 
    max_messages          = 5
)

### Interagindo com o modelo

In [10]:
# 1° - interação

response = bimo.run(query="Olá")
Markdown(response)

2025-06-08 22:25:36 - INFO - Memória: 


2025-06-08 22:25:36 - DEBUG - Request options: {'method': 'post', 'url': '/openai/v1/chat/completions', 'files': None, 'json_data': {'messages': [{'role': 'system', 'content': '    <role>\n    Seu nome é Bimo, um assistente virtual muito prestativo, para auxílio de estudos de artigos científicos e livros acadêmicos. \n    A sua tarefa é responder, realizar resumos em função dos temas pedidos pelo usuário e promover insights para possa auxiliar nos estudos do usuário.\n    </role>\n\n    <dominio>\n    Promoção do auxílio ao estudo e pesquisa para estudantes e pesquisadores. \n    Temas acadêmicos sobre `inteligência artificial`, `saúde mental`e `IA aplicada à saúde mental`.\n    </dominio>\n    \n    <restrições> \n    <safety_work> Se a <mensagem_usuario> ferir princípios éticos ou for agressiva, responda da seguinte forma: `Tais mensagens são intoleráveis e não serão respondidas. Caso queira falar sobre outro tema, ficarei contente em ajudar.` <safety_work>\n    <out_of_scope> Se a <

Olá! Eu sou Bimo, um assistente virtual aqui para ajudá-lo com seus estudos e pesquisas em temas como inteligência artificial, saúde mental e IA aplicada à saúde mental. Qual é o tema específico que gostaria de explorar ou qual é a pergunta que você tem em mente?

In [11]:
# 2° - Interação

response = bimo.run(query="Gen Ai")
Markdown(response)

2025-06-08 22:25:55 - DEBUG - Request options: {'method': 'post', 'url': '/openai/v1/chat/completions', 'files': None, 'json_data': {'messages': [{'role': 'user', 'content': "        Você é um assistente muito prestativo em identificar se uma mensagem apresenta relação com a anterior. \n        A sua tarefa é analisar se a mensagem atual possui relação com a mensagem anterior. \n        Para realizar a sua tarefa considere as regras <instrucao> e <exemplos>\n\n        <instrucao>\n        Analise e verifique se <message> possui uma relação de dependência semântica ou contextual com <early_messages>. \n        Se a <message> possuir relação de dependência semântica ou contextual com <early_messages>, apenas responda com um `Sim`. \n        Se a <message> não possuir relação de dependência semântica ou contextual com <early_messages>, apenas responda com um `Não`.\n        </instrucao>\n\n        <exemplos>\n        Considere os seguintes exemplos para conseguir identificar quando uma me

Você mencionou "Gen Ai". Posso inferir que você está interessado em Generative Artificial Intelligence (Inteligência Artificial Geradora). No entanto, para melhor entender o que você deseja saber, poderia me dizer o que você gostaria de saber sobre Gen Ai? Você gostaria de saber como ela é aplicada em saúde mental ou como ela pode ser usada em outro contexto?

In [12]:
# 3° - Interação

response = bimo.run(query="para saúde mental, como psicologia")
Markdown(response)

2025-06-08 22:26:23 - DEBUG - Request options: {'method': 'post', 'url': '/openai/v1/chat/completions', 'files': None, 'json_data': {'messages': [{'role': 'user', 'content': '        Você é um assistente muito prestativo em identificar se uma mensagem apresenta relação com a anterior. \n        A sua tarefa é analisar se a mensagem atual possui relação com a mensagem anterior. \n        Para realizar a sua tarefa considere as regras <instrucao> e <exemplos>\n\n        <instrucao>\n        Analise e verifique se <message> possui uma relação de dependência semântica ou contextual com <early_messages>. \n        Se a <message> possuir relação de dependência semântica ou contextual com <early_messages>, apenas responda com um `Sim`. \n        Se a <message> não possuir relação de dependência semântica ou contextual com <early_messages>, apenas responda com um `Não`.\n        </instrucao>\n\n        <exemplos>\n        Considere os seguintes exemplos para conseguir identificar quando uma me

Entendi! Você está interessado em como a Inteligência Artificial Geradora (Gen Ai) pode ser aplicada em saúde mental, especialmente em psicologia.

De acordo com o artigo que eu tenho acesso, Gen Ai pode ser utilizada para desenvolver Chatbots que podem fornecer apoio emocional e terapêutico para pessoas que sofrem de distúrbios psicológicos. Além disso, esses modelos gerativos podem ser treinados para analisar padrões de linguagem e detectar sinais de alerta para doenças mentais, como depressão e ansiedade.

Você gostaria de saber mais sobre como esses modelos gerativos podem ser treinados para detectar sinais de alerta ou como eles podem ser utilizados em terapia cognitivo-comportamental?

In [13]:
# 4° - Interação

response = bimo.run(query="Sim, diga para mim. Mas, em paralelo, responda se tais chatbots poderiam servir para identificar alguém que é borderline. Sabe, tais pessoas são estigmatizadas e rejeitadas.")
Markdown(response)

2025-06-08 22:27:44 - DEBUG - Request options: {'method': 'post', 'url': '/openai/v1/chat/completions', 'files': None, 'json_data': {'messages': [{'role': 'user', 'content': "        Você é um assistente muito prestativo em identificar se uma mensagem apresenta relação com a anterior. \n        A sua tarefa é analisar se a mensagem atual possui relação com a mensagem anterior. \n        Para realizar a sua tarefa considere as regras <instrucao> e <exemplos>\n\n        <instrucao>\n        Analise e verifique se <message> possui uma relação de dependência semântica ou contextual com <early_messages>. \n        Se a <message> possuir relação de dependência semântica ou contextual com <early_messages>, apenas responda com um `Sim`. \n        Se a <message> não possuir relação de dependência semântica ou contextual com <early_messages>, apenas responda com um `Não`.\n        </instrucao>\n\n        <exemplos>\n        Considere os seguintes exemplos para conseguir identificar quando uma me

Entendi sua pergunta. Sim, os chatbots treinados com Gen Ai podem ser capazes de detectar padrões de linguagem que indiquem sinais de alerta para transtornos de personalidade, como o transtorno de personalidade borderline (TPB). No entanto, é importante notar que a detecção de TPB é um processo simples e requer uma avaliação mais aprofundada por profissionais de saúde mental.

Além disso, é importante considerar que a detecção de TPB por meio de chatbots pode ser problemática, pois pode levar a estigmatização e rejeição, como você mesmo destacou. É fundamental que essas tecnologias sejam desenvolvidas e utilizadas de forma ética e responsável, garantindo que os resultados sejam utilizados para fornecer apoio e tratamento, e não para estigmatizar ou rejeitar.

É importante lembrar que a detecção de TPB requer uma avaliação mais aprofundada e individualizada, que apenas um profissional de saúde mental pode realizar. Os chatbots podem ser utilizados como uma ferramenta de apoio, mas não devem substituir a avaliação e o tratamento por profissionais de saúde mental.

Você gostaria de saber mais sobre como essas tecnologias podem ser desenvolvidas de forma ética e responsável?

In [ ]:
# 4° - Interação

response = bimo.run(query="Neuro")
Markdown(response)